# 📊 Chapter 7: Monitoring and Logging

📖 Book: Chapter 7, pp. 263–306 · ⏱️ 2–3 hours · 🧱 Needs: Ch 1–2, 6 · Level: Intermediate

---

> **🎯 Key idea:** Structured JSON `logging`, `RotatingFileHandler`, Prometheus `/metrics/`, and OpenTelemetry give you visibility before users report outages.

> **💡 Tip:** Run cells sequentially top-to-bottom (`Shift + Enter`).

> **⚠️ Warning:** Cells tagged `external` may require extra packages; cloud stacks incur costs if deployed.

## 🔄 Then vs. Now

> **📜 Then (book, ~2020):** Manual log correlation / Graphite  
> **🆕 Now (2026):** OpenTelemetry (OTel) + Prometheus + Grafana  
> **Why:** Vendor-neutral logs/metrics/traces

> **📜 Then (book, ~2020):** `%` formatting & plain text  
> **🆕 Now (2026):** Structured JSON + lazy `%s` logger args  
> **Why:** Machine-queryable in ELK/Loki

---

## 4. 💡 Conceptual Cards

### 🚨 Real-World Problem
A microservice crashes in production, and all you have are scattered `print()` lines with no timestamps or severity levels—or worse, unrotated logs filled the root disk. How do you record structured events safely and know when error rates spike before customers call? You configure Python `logging`, JSON formatters, and Prometheus/StatsD metrics.

### Concept: Structured `logging`
**What:** Stdlib `dictConfig` + `RotatingFileHandler`  
**Why:** `print()` lacks levels, rotation & thread safety  
**How:** Configure once at startup; inject env via `Filter`  
**When:** All production Python services

📖 *Book listing:* `EnvironFilter` `dictConfig` Logger (pp. 288–296)

**Plain-language analogy:** Using `print()` in production is like shouting into a hallway; `logging` is a structured flight recorder with timestamps, severity levels, and automatic file rotation.

**Walkthrough:** The book configures Python `logging` once at startup using `dictConfig`, attaches a custom `logging.Filter` to inject `APP_ENVIRON`, and logs tracebacks via `logging.exception()`.

### Concept: Push vs. Pull Metrics
**What:** StatsD (UDP push) vs. Prometheus (HTTP pull)  
**Why:** Avoid blocking app threads; detect down targets  
**How:** StatsD batches UDP; Prometheus scrapes `/metrics/`  
**When:** High-throughput apps & K8s

📖 *Book listing:* Flask `/metrics/` Prometheus Exporter (pp. 278–284)

**Plain-language analogy:** StatsD (push) is like dropping postcards in a mailbox without waiting; Prometheus (pull) is like a nurse checking each patient's vitals on a regular round.

**Walkthrough:** The book instruments a Flask app with `Counter` and `Histogram` timers, exposing `prometheus_client.generate_latest()` on `/metrics/` for Prometheus to scrape over HTTP.

## 6. 🟢 Beginner Examples

#### Example 1: Beginner Example 1: Module logger & severity levels
*Expected output:* Printed labelled output below.

In [1]:
# Beginner Example 1: Module logger & severity levels
import logging

logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(name)s: %(message)s', force=True)
log = logging.getLogger('deploy.svc')
log.debug('Debug hidden at INFO')
log.info('Deploying %s', 'api-v2')
log.warning('Disk at %d%%', 82)

[INFO] deploy.svc: Deploying api-v2


[WARNING] deploy.svc: Disk at 82%


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 2: Beginner Example 2: Capturing tracebacks with logging.exception()
*Expected output:* Printed labelled output below.

In [2]:
# Beginner Example 2: Capturing tracebacks with logging.exception()
try:
    _ = 1 / 0
except ZeroDivisionError:
    log.exception('Metric division failed')

[ERROR] deploy.svc: Metric division failed
Traceback (most recent call last):
  File "/tmp/ipykernel_341889/581668544.py", line 3, in <module>
    _ = 1 / 0
        ~~^~~
ZeroDivisionError: division by zero


**What the output means:** Shows the exact exception caught cleanly instead of crashing the whole script.

> **🧪 Try it:** Change the invalid input to a valid one and observe the `except` branch no longer triggers.

#### Example 3: Beginner Example 3: Hierarchical metric naming helper
*Expected output:* Printed labelled output below.

In [3]:
# Beginner Example 3: Hierarchical metric naming helper
def build_metric_name(region: str, env: str, service: str, metric: str) -> str:
    return '.'.join(p.strip().lower() for p in (region, env, service, metric))

print('Metric:', build_metric_name('US-EAST-1', 'PROD', 'Auth', 'Errors'))

Metric: us-east-1.prod.auth.errors


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

## 7. 🟡 Intermediate Examples: `dictConfig`, `RotatingFileHandler`, JSON Logs & Prometheus

#### Example 4: Intermediate Example 1: dictConfig + EnvironFilter + RotatingFileHandler
*Expected output:* Printed labelled output below.

In [4]:
# Intermediate Example 1: dictConfig + EnvironFilter + RotatingFileHandler
import logging, os, tempfile
from logging.config import dictConfig
from pathlib import Path

class EnvironFilter(logging.Filter):
    def filter(self, record):
        record.app_env = os.environ.get('APP_ENVIRON', 'DEV')
        return True

with tempfile.TemporaryDirectory() as tmp:
    lf = Path(tmp) / 'app.log'
    dictConfig({
        'version': 1,
        'filters': {'env': {'()': EnvironFilter}},
        'formatters': {'std': {'format': '[%(app_env)s][%(levelname)s] %(message)s'}},
        'handlers': {
            'console': {'class': 'logging.StreamHandler', 'formatter': 'std', 'filters': ['env']},
            'file': {'class': 'logging.handlers.RotatingFileHandler', 'filename': str(lf), 'maxBytes': 1024, 'backupCount': 2, 'formatter': 'std', 'filters': ['env']},
        },
        'root': {'level': 'INFO', 'handlers': ['console', 'file']},
    })
    logging.info('Rotating logger ready')
    print('File output:', lf.read_text().strip())
    logging.shutdown()

[DEV][INFO] Rotating logger ready


File output: [DEV][INFO] Rotating logger ready


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 5: Intermediate Example 2 (External + Mock Fallback): Prometheus & OpenTelemetry
*Expected output:* Simulated fallback if optional package is missing.

In [5]:
# Intermediate Example 2 (External + Mock Fallback): Prometheus & OpenTelemetry
# Requires: pip install flask prometheus_client opentelemetry-api
try:
    from prometheus_client import Counter, generate_latest
    c = Counter('demo_requests_total', 'Demo requests')
    c.inc()
    print('Prometheus sample:', generate_latest().decode().splitlines()[-1])
except ImportError:
    print('prometheus_client not installed; simulated counter: demo_requests_total 1.0')

Prometheus sample: demo_requests_created 1.7957702475187347e+09


**What the output means:** If the optional library is installed, it runs live; otherwise the `except ImportError:` fallback prints a safe offline simulation.

> **🧪 Try it:** Install the optional package in your `.venv` (see the comment at top of the cell) and re-run to compare live vs. simulated output.

## 8. 🚀 Real-World DevOps Applications (Personal DevOps Learning Track)
*Supplementary extension:* **Structured JSON Log Formatter**—ready for Elasticsearch, Loki, or CloudWatch Logs Insights.

#### Example 6: import json, logging
*Expected output:* Printed labelled output below.

In [6]:
import json, logging

class JsonFormatter(logging.Formatter):
    def format(self, record: logging.LogRecord) -> str:
        return json.dumps({'level': record.levelname, 'logger': record.name, 'msg': record.getMessage()})

jlog = logging.getLogger('json.demo')
jlog.handlers.clear()
jlog.propagate = False
h = logging.StreamHandler()
h.setFormatter(JsonFormatter())
jlog.addHandler(h)
jlog.setLevel(logging.INFO)
jlog.info('Health check passed')

{"level": "INFO", "logger": "json.demo", "msg": "Health check passed"}


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Real Tool Demo (guarded with `shutil.which`)
*Expected output:* CLI version or check output (or graceful skip).

In [7]:
# Real Tool Demo: curl CLI check (guarded with shutil.which)
import shutil, subprocess
if shutil.which('curl'):
    print('curl:', subprocess.run(['curl', '--version'], capture_output=True, text=True, check=True).stdout.splitlines()[0])
else:
    print('curl not installed; skipped')

curl: curl 8.18.0 (x86_64-pc-linux-gnu) libcurl/8.18.0 OpenSSL/3.5.5 zlib/1.3.1 brotli/1.2.0 zstd/1.5.7 libidn2/2.3.8 libpsl/0.21.2 libssh2/1.11.1 nghttp2/1.68.0 librtmp/2.3 mit-krb5/1.22.1 OpenLDAP/2.6.10


**What the output means:** Checks whether the real CLI tool exists on `PATH`; if found, runs a safe read-only/dry-run check, otherwise skips cleanly.

> **🧪 Try it:** Try adding `--help` or inspecting `cp.returncode` to see how exit codes behave.

---

## 9. ⚠️ Common Mistakes & Troubleshooting

- ❌ **Duplicate log lines** -> `addHandler()` + `propagate=True`. ✅ Clear handlers or set `propagate = False`.
- ❌ **Eager f-strings in `debug()`** -> Formats strings even when suppressed. ✅ Use lazy `logger.debug('x=%s', x)`.
- ❌ **Unbounded `FileHandler`** -> Fills root disk. ✅ Use `RotatingFileHandler` or `stdout`.

#### Example 7: Prevent duplicate log lines by clearing handlers and setting propagate = False
*Expected output:* Printed labelled output below.

In [8]:
# Prevent duplicate log lines by clearing handlers and setting propagate = False
print('Handlers on jlog:', len(jlog.handlers), '| propagate:', jlog.propagate)

Handlers on jlog: 1 | propagate: False


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

## 10. 🛠️ Hands-On Exercises
1. **(Beginner)** `build_metric_name(region, env, svc, metric)` returning lowercase dot-separated string.
2. **(Beginner)** Configure a logger at `WARNING` and verify `INFO` is suppressed.
3. **(Beginner)** Log a caught `ValueError` with `logging.exception()`.
4. **(Intermediate)** Extend `JsonFormatter` to include `timestamp` via `self.formatTime(record)`.
5. **(Intermediate)** Build `MetricRegistry` with `.inc(name)` and `.render_prometheus()`.

## 11. 🏆 Practical DevOps Challenge
`check_services(endpoints: list[str]) -> dict`: Log each endpoint check in JSON, increment `checks_total` and `checks_failed`, and return Prometheus-formatted metrics.

> **🧪 Try it:** The starter cell below reports `TODO` until you implement the function body.

> **🧪 Try it:** Run the starter cell below—it prints `TODO` until you implement the body.

#### 🧪 Starter Exercise: `Starter Exercise Cell (implement build_metric_name and run to verify!)`
*Expected output:* `TODO` message until implemented, then `✅ Starter checks PASSED!`.

In [9]:
# Starter Exercise Cell (implement build_metric_name and run to verify!)
def _run_starter_checks():
    assert build_metric_name('US', 'PROD', 'Web', '200') == 'us.prod.web.200'
def build_metric_name(region: str, env: str, service: str, metric: str) -> str:
    raise NotImplementedError('Implement build_metric_name')

try:
    _run_starter_checks()
    print('✅ Starter checks PASSED!')
except NotImplementedError as e:
    print(f'🧪 TODO: {e}')

🧪 TODO: Implement build_metric_name


## 12. 🧠 Knowledge Checks (Inline Q&A for Practice)
**Q1:** Why is `logging.error('err %s', x)` faster than `logging.error(f'err {x}')`?

<details>
<summary>✅ Answer 1</summary>

`%s` defers string formatting until the log record actually passes the level filter.

</details>

**Q2:** What does `logging.exception()` add?

<details>
<summary>✅ Answer 2</summary>

Automatically attaches the active exception stack trace.

</details>

**Q3:** Why prefer `RotatingFileHandler` over `FileHandler`?

<details>
<summary>✅ Answer 3</summary>

Caps file size and rotates backups so logs never fill the disk.

</details>

**Q4:** How do Prometheus (pull) and StatsD (push) differ?

<details>
<summary>✅ Answer 4</summary>

Prometheus scrapes `/metrics/` over HTTP; apps fire non-blocking UDP packets to StatsD.

</details>

**Q5:** What does Logstash `grok` do?

<details>
<summary>✅ Answer 5</summary>

Parses unstructured text logs into structured JSON fields.

</details>

## 13. 🔑 Hints & Complete Solutions

<details>
<summary>💭 Hints (click to expand)</summary>

- **Ex 5**: Store counters in `dict[str, int]` and return `'\n'.join(f'{k} {v}' for ...)`.

</details>

> **💡 Tip:** Complete solutions are in the collapsed/marked cell below.

<details>
<summary>✅ Complete Solutions (expand/run cell below to verify)</summary>

Run the verified solutions cell below to check all exercise & challenge assertions.

</details>

In [10]:
# Complete Solutions + Assert Verification
import json, logging

def build_metric_name(region: str, env: str, service: str, metric: str) -> str:
    return '.'.join(p.strip().lower() for p in (region, env, service, metric))

class TimestampJsonFormatter(logging.Formatter):
    def format(self, record: logging.LogRecord) -> str:
        return json.dumps({'ts': self.formatTime(record), 'level': record.levelname, 'msg': record.getMessage()})

class MetricRegistry:
    def __init__(self) -> None:
        self._counters: dict[str, int] = {}
    def inc(self, name: str, amount: int = 1) -> None:
        self._counters[name] = self._counters.get(name, 0) + amount
    def render_prometheus(self) -> str:
        return '\n'.join(f'{k} {v}' for k, v in sorted(self._counters.items()))

def check_services(endpoints: list[str]) -> dict:
    reg = MetricRegistry()
    for ep in endpoints:
        reg.inc('checks_total')
        if 'down' in ep:
            reg.inc('checks_failed')
    return {'metrics': reg.render_prometheus()}

assert build_metric_name('US', 'PROD', 'Web', '200') == 'us.prod.web.200'
out = check_services(['https://api/ok', 'https://db/down'])['metrics']
assert 'checks_failed 1' in out and 'checks_total 2' in out
print('All Chapter 7 assertions PASSED!')

All Chapter 7 assertions PASSED!


---

## 14. 📌 Chapter Summary & Quick Reference

- `logging.config.dictConfig()` — Centralized logger config (e.g. `dictConfig({'version': 1, ...})`)
- `logger.exception('msg')` — Log error + stack trace (e.g. Inside `except:` blocks)
- `Counter` / `Histogram` — Prometheus instrumentation (e.g. `REQUESTS.labels('/').inc()`)

### ✅ What You Can Now Do
- Configure `logging.config.dictConfig()` with `RotatingFileHandler` and custom `Filter`s.
- Emit structured JSON logs and capture tracebacks cleanly with `logging.exception()`.
- Instrument services with Prometheus counters/histograms and hierarchical metric names.

### ➡️ Next Chapter
Chapter 8 builds on this by writing automated `pytest` and `testinfra` tests to verify your code and infrastructure.